In [2]:
!ls -lh ../data/processed

total 1193776
-rw-r--r--@ 1 bhavingaikwad  staff   583M Sep  2 09:16 nyc_sales_2003_2025_combined.csv


In [1]:
import pandas as pd
import re

combined = pd.read_csv("../data/processed/nyc_sales_2003_2025_combined.csv", low_memory=False)
combined['SALE DATE'] = pd.to_datetime(combined['SALE DATE'])
combined['BBL'] = combined['BBL'].astype('Int64')

print(combined.shape)

(2070484, 29)


In [2]:
import re

def normalize_address(addr):
    if pd.isna(addr):
        return None
    addr = addr.upper().strip()
    addr = re.sub(r'\s+', ' ', addr)  # collapse multiple spaces
    replacements = {
        r'\bSTREET\b': 'ST', r'\bAVENUE\b': 'AVE', r'\bBOULEVARD\b': 'BLVD',
        r'\bPLACE\b': 'PL', r'\bROAD\b': 'RD', r'\bDRIVE\b': 'DR',
        r'\bLANE\b': 'LN', r'\bCOURT\b': 'CT', r'\bPARKWAY\b': 'PKWY',
        r'\bSQUARE\b': 'SQ',
    }
    for pattern, repl in replacements.items():
        addr = re.sub(pattern, repl, addr)
    return addr

# Test on real, known values from OUR data before trusting it at scale
test_samples = combined['ADDRESS'].dropna().sample(15, random_state=42).tolist()
for a in test_samples:
    print(f"{a!r:50} -> {normalize_address(a)!r}")

'2048 EAST 17TH STREET'                            -> '2048 EAST 17TH ST'
'121 KINGSLAND AVENUE                     '        -> '121 KINGSLAND AVE'
'81 OCEAN PARKWAY, 5M'                             -> '81 OCEAN PKWY, 5M'
'107-12 125TH STREET'                              -> '107-12 125TH ST'
'34 MARKHAM PLACE                         '        -> '34 MARKHAM PL'
'1911 HAIGHT AVENUE'                               -> '1911 HAIGHT AVE'
'1644 73 STREET                           '        -> '1644 73 ST'
'34-46 201 STREET                         '        -> '34-46 201 ST'
'27-13 41ST   AVENUE                      '        -> '27-13 41ST AVE'
'2379 EAST 2ND STREET                     '        -> '2379 EAST 2ND ST'
'7313 71 STREET                           '        -> '7313 71 ST'
'1335 AVENUE OF THE AMER                  '        -> '1335 AVE OF THE AMER'
'16 WEST 16TH ST, 6H-S'                            -> '16 WEST 16TH ST, 6H-S'
'5425 VALLES AVENUE, 1B                   '        -> '5425 VA

In [3]:
edge_cases = [
    "73RD STREET", "73 STREET", "73rd St",
    "1ST AVENUE", "FIRST AVENUE",
    "  123  MAIN   ST  ",
]
for a in edge_cases:
    print(f"{a!r:30} -> {normalize_address(a)!r}")

'73RD STREET'                  -> '73RD ST'
'73 STREET'                    -> '73 ST'
'73rd St'                      -> '73RD ST'
'1ST AVENUE'                   -> '1ST AVE'
'FIRST AVENUE'                 -> 'FIRST AVE'
'  123  MAIN   ST  '           -> '123 MAIN ST'


In [4]:
def normalize_address(addr):
    if pd.isna(addr):
        return None
    addr = addr.upper().strip()
    addr = re.sub(r'\s+', ' ', addr)
    
    # Strip ordinal suffix only when it directly follows digits (word-boundaried)
    addr = re.sub(r'\b(\d+)(ST|ND|RD|TH)\b', r'\1', addr)
    
    replacements = {
        r'\bSTREET\b': 'ST', r'\bAVENUE\b': 'AVE', r'\bBOULEVARD\b': 'BLVD',
        r'\bPLACE\b': 'PL', r'\bROAD\b': 'RD', r'\bDRIVE\b': 'DR',
        r'\bLANE\b': 'LN', r'\bCOURT\b': 'CT', r'\bPARKWAY\b': 'PKWY',
        r'\bSQUARE\b': 'SQ',
    }
    for pattern, repl in replacements.items():
        addr = re.sub(pattern, repl, addr)
    return addr

# Re-test the ordinal cases we already found problematic
edge_cases = [
    "73RD STREET", "73 STREET", "73rd St",
    "1ST AVENUE", "FIRST AVENUE",
    "  123  MAIN   ST  ",
]
print("--- Original edge cases ---")
for a in edge_cases:
    print(f"{a!r:30} -> {normalize_address(a)!r}")

# NEW: specifically hunt for collision risk (ordinal suffix NOT in a street number context)
collision_tests = [
    "123 MAIN ST, 3RD FLOOR",
    "45 PARK AVE APT 2ND FL",
    "10 BROADWAY, 21ST FLOOR",
    "5 AVE, SUITE 3RD",
]
print("\n--- Collision risk tests ---")
for a in collision_tests:
    print(f"{a!r:30} -> {normalize_address(a)!r}")

# Re-run full sample to confirm nothing else broke
print("\n--- Original 15-row sample, re-verified ---")
for a in test_samples:
    print(f"{a!r:50} -> {normalize_address(a)!r}")

--- Original edge cases ---
'73RD STREET'                  -> '73 ST'
'73 STREET'                    -> '73 ST'
'73rd St'                      -> '73 ST'
'1ST AVENUE'                   -> '1 AVE'
'FIRST AVENUE'                 -> 'FIRST AVE'
'  123  MAIN   ST  '           -> '123 MAIN ST'

--- Collision risk tests ---
'123 MAIN ST, 3RD FLOOR'       -> '123 MAIN ST, 3 FLOOR'
'45 PARK AVE APT 2ND FL'       -> '45 PARK AVE APT 2 FL'
'10 BROADWAY, 21ST FLOOR'      -> '10 BROADWAY, 21 FLOOR'
'5 AVE, SUITE 3RD'             -> '5 AVE, SUITE 3'

--- Original 15-row sample, re-verified ---
'2048 EAST 17TH STREET'                            -> '2048 EAST 17 ST'
'121 KINGSLAND AVENUE                     '        -> '121 KINGSLAND AVE'
'81 OCEAN PARKWAY, 5M'                             -> '81 OCEAN PKWY, 5M'
'107-12 125TH STREET'                              -> '107-12 125 ST'
'34 MARKHAM PLACE                         '        -> '34 MARKHAM PL'
'1911 HAIGHT AVENUE'                               -

In [5]:
def normalize_address(addr):
    if pd.isna(addr):
        return None
    addr = addr.upper().strip()
    addr = re.sub(r'\s+', ' ', addr)
    
    replacements = {
        r'\bSTREET\b': 'ST', r'\bAVENUE\b': 'AVE', r'\bBOULEVARD\b': 'BLVD',
        r'\bPLACE\b': 'PL', r'\bROAD\b': 'RD', r'\bDRIVE\b': 'DR',
        r'\bLANE\b': 'LN', r'\bCOURT\b': 'CT', r'\bPARKWAY\b': 'PKWY',
        r'\bSQUARE\b': 'SQ',
    }
    for pattern, repl in replacements.items():
        addr = re.sub(pattern, repl, addr)
    
    # Strip ordinal suffix ONLY when immediately followed by a street-type word
    street_types = r'(?:ST|AVE|BLVD|PL|RD|DR|LN|CT|PKWY|SQ)'
    addr = re.sub(rf'\b(\d+)(ST|ND|RD|TH)\b(?=\s+{street_types}\b)', r'\1', addr)
    
    return addr

print("--- Collision risk tests (should be UNCHANGED now) ---")
for a in collision_tests:
    print(f"{a!r:30} -> {normalize_address(a)!r}")

print("\n--- Original edge cases (should still strip correctly) ---")
for a in edge_cases:
    print(f"{a!r:30} -> {normalize_address(a)!r}")

print("\n--- Original 15-row sample ---")
for a in test_samples:
    print(f"{a!r:50} -> {normalize_address(a)!r}")

--- Collision risk tests (should be UNCHANGED now) ---
'123 MAIN ST, 3RD FLOOR'       -> '123 MAIN ST, 3RD FLOOR'
'45 PARK AVE APT 2ND FL'       -> '45 PARK AVE APT 2ND FL'
'10 BROADWAY, 21ST FLOOR'      -> '10 BROADWAY, 21ST FLOOR'
'5 AVE, SUITE 3RD'             -> '5 AVE, SUITE 3RD'

--- Original edge cases (should still strip correctly) ---
'73RD STREET'                  -> '73 ST'
'73 STREET'                    -> '73 ST'
'73rd St'                      -> '73 ST'
'1ST AVENUE'                   -> '1 AVE'
'FIRST AVENUE'                 -> 'FIRST AVE'
'  123  MAIN   ST  '           -> '123 MAIN ST'

--- Original 15-row sample ---
'2048 EAST 17TH STREET'                            -> '2048 EAST 17 ST'
'121 KINGSLAND AVENUE                     '        -> '121 KINGSLAND AVE'
'81 OCEAN PARKWAY, 5M'                             -> '81 OCEAN PKWY, 5M'
'107-12 125TH STREET'                              -> '107-12 125 ST'
'34 MARKHAM PLACE                         '        -> '34 MARKHAM PL'


In [6]:
combined['ADDRESS_NORM'] = combined['ADDRESS'].apply(normalize_address)

# Sanity checks at full scale
print(f"Nulls in ADDRESS_NORM: {combined['ADDRESS_NORM'].isnull().sum()}")  # should match null count in ADDRESS
print(f"Nulls in ADDRESS: {combined['ADDRESS'].isnull().sum()}")
print(f"Unique normalized addresses: {combined['ADDRESS_NORM'].nunique():,}")
print(f"Unique raw addresses: {combined['ADDRESS'].nunique():,}")

Nulls in ADDRESS_NORM: 0
Nulls in ADDRESS: 0
Unique normalized addresses: 1,055,372
Unique raw addresses: 1,364,778


In [7]:
block_bbl = combined.groupby(['BOROUGH', 'BLOCK', 'LOT']).size().reset_index(name='n_sales')

print(f"Total blocking groups: {len(block_bbl):,}")
print(f"Groups with 2+ sales (candidate repeat-sales): {(block_bbl['n_sales'] >= 2).sum():,}")
print(block_bbl['n_sales'].value_counts().sort_index().head(15))

Total blocking groups: 863,366
Groups with 2+ sales (candidate repeat-sales): 450,235
n_sales
1     413131
2     248022
3     116558
4      46860
5      18806
6       7850
7       3478
8       1639
9        858
10       550
11       313
12       247
13       161
14       179
15       157
Name: count, dtype: int64


In [8]:
block_addr = combined.groupby(['ZIP CODE', 'ADDRESS_NORM']).size().reset_index(name='n_sales')

print(f"Total blocking groups: {len(block_addr):,}")
print(f"Groups with 2+ sales (candidate repeat-sales): {(block_addr['n_sales'] >= 2).sum():,}")
print(block_addr['n_sales'].value_counts().sort_index().head(15))

Total blocking groups: 1,066,572
Groups with 2+ sales (candidate repeat-sales): 393,672
n_sales
1     672900
2     235314
3      88934
4      34053
5      14065
6       6793
7       3441
8       2017
9       1212
10       886
11       628
12       519
13       442
14       363
15       367
Name: count, dtype: int64


In [9]:
from rapidfuzz import fuzz

# Within each ZIP code, compare all distinct addresses pairwise to find near-matches
# (not identical strings, but similar enough to likely be the same property)

sample_zip = combined[combined['ZIP CODE'] == 11214]  # Bath Beach area, from an earlier example
unique_addrs = sample_zip['ADDRESS_NORM'].dropna().unique()

print(f"Unique normalized addresses in this ZIP: {len(unique_addrs)}")

# Compare a few pairs to see what similarity scores look like on REAL near-duplicates
import itertools
near_matches = []
for a, b in itertools.combinations(unique_addrs, 2):
    score = fuzz.token_sort_ratio(a, b)
    if 80 <= score < 100:  # similar but not identical
        near_matches.append((a, b, score))

near_matches.sort(key=lambda x: -x[2])
for a, b, score in near_matches[:15]:
    print(f"{score:.0f}  {a!r:40} <-> {b!r}")

Unique normalized addresses in this ZIP: 9903
98  '2630 CROPSEY AVE, APT 8E'               <-> '2630 CROPSEY AVE, APT 8'
98  '2630 CROPSEY AVE, 14A'                  <-> '2630 CROPSEY AVE, 1A'
98  '2630 CROPSEY AVE, 1G'                   <-> '2630 CROPSEY AVE, 11G'
98  '2630 CROPSEY AVE, 1G'                   <-> '2630 CROPSEY AVE, 12G'
98  '2630 CROPSEY AVE, 1G'                   <-> '2630 CROPSEY AVE, 13G'
98  '2630 CROPSEY AVE, 1C'                   <-> '2630 CROPSEY AVE, 12C'
98  '2630 CROPSEY AVE, 1C'                   <-> '2630 CROPSEY AVE, 15C'
98  '2630 CROPSEY AVE, 1C'                   <-> '2630 CROPSEY AVE, 14C'
98  '2630 CROPSEY AVE, 1C'                   <-> '2630 CROPSEY AVE, 13C'
98  '2630 CROPSEY AVE, 1C'                   <-> '2630 CROPSEY AVE, 16C'
98  '2630 CROPSEY AVE, 16H'                  <-> '2630 CROPSEY AVE, 1H'
98  '2630 CROPSEY AVE, 15D'                  <-> '2630 CROPSEY AVE, 1D'
98  '2630 CROPSEY AVE, 1F'                   <-> '2630 CROPSEY AVE, 14F'
98  '2

In [10]:
def split_address(addr):
    if pd.isna(addr):
        return None, None
    if ',' in addr:
        base, unit = addr.split(',', 1)
        return base.strip(), unit.strip()
    return addr.strip(), None

test_cases = [
    '2630 CROPSEY AVE, 1G',
    '2630 CROPSEY AVE, 11G',
    '2630 CROPSEY AVE, APT 8E',
    '2630 CROPSEY AVE, APT 8',
    '73 ST',  # no comma at all, should return (base, None)
]
for a in test_cases:
    print(f"{a!r:35} -> base={split_address(a)[0]!r}, unit={split_address(a)[1]!r}")

'2630 CROPSEY AVE, 1G'              -> base='2630 CROPSEY AVE', unit='1G'
'2630 CROPSEY AVE, 11G'             -> base='2630 CROPSEY AVE', unit='11G'
'2630 CROPSEY AVE, APT 8E'          -> base='2630 CROPSEY AVE', unit='APT 8E'
'2630 CROPSEY AVE, APT 8'           -> base='2630 CROPSEY AVE', unit='APT 8'
'73 ST'                             -> base='73 ST', unit=None


In [11]:
def normalize_unit(unit):
    if unit is None:
        return None
    unit = unit.upper().strip()
    unit = re.sub(r'^(APT|APARTMENT|UNIT|#)\.?\s*', '', unit)
    return unit.strip() if unit else None

test_units = ['1G', '11G', 'APT 8E', 'APT 8', '#4B', 'UNIT 12', '']
for u in test_units:
    print(f"{u!r:15} -> {normalize_unit(u)!r}")

'1G'            -> '1G'
'11G'           -> '11G'
'APT 8E'        -> '8E'
'APT 8'         -> '8'
'#4B'           -> '4B'
'UNIT 12'       -> '12'
''              -> None


In [12]:
combined[['ADDRESS_BASE', 'ADDRESS_UNIT']] = combined['ADDRESS_NORM'].apply(
    lambda x: pd.Series(split_address(x))
)
combined['ADDRESS_UNIT'] = combined['ADDRESS_UNIT'].apply(normalize_unit)

print(combined[['ADDRESS_NORM', 'ADDRESS_BASE', 'ADDRESS_UNIT']].sample(10, random_state=1))

AttributeError: 'float' object has no attribute 'upper'

In [13]:
def normalize_unit(unit):
    if pd.isna(unit):
        return None
    unit = str(unit).upper().strip()
    unit = re.sub(r'^(APT|APARTMENT|UNIT|#)\.?\s*', '', unit)
    return unit.strip() if unit else None

test_units = ['1G', '11G', 'APT 8E', 'APT 8', '#4B', 'UNIT 12', '', None, float('nan')]
for u in test_units:
    print(f"{u!r:15} -> {normalize_unit(u)!r}")

'1G'            -> '1G'
'11G'           -> '11G'
'APT 8E'        -> '8E'
'APT 8'         -> '8'
'#4B'           -> '4B'
'UNIT 12'       -> '12'
''              -> None
None            -> None
nan             -> None


In [14]:
combined[['ADDRESS_BASE', 'ADDRESS_UNIT']] = combined['ADDRESS_NORM'].apply(
    lambda x: pd.Series(split_address(x))
)
combined['ADDRESS_UNIT'] = combined['ADDRESS_UNIT'].apply(normalize_unit)

print(combined[['ADDRESS_NORM', 'ADDRESS_BASE', 'ADDRESS_UNIT']].sample(10, random_state=1))
print(f"\nNull ADDRESS_UNIT count: {combined['ADDRESS_UNIT'].isnull().sum():,}")

                   ADDRESS_NORM       ADDRESS_BASE ADDRESS_UNIT
546893          608 WEST 149 ST    608 WEST 149 ST          NaN
225168        455 BAINBRIDGE ST  455 BAINBRIDGE ST          NaN
1007353            132-35 41 RD       132-35 41 RD          NaN
654448   1 SUTTON PL SOUTH, 12D  1 SUTTON PL SOUTH          12D
901543            109-33 46 AVE      109-33 46 AVE          NaN
459680                      220                220          NaN
182280                215 ADAMS          215 ADAMS          NaN
357619           103 QUENTIN RD     103 QUENTIN RD          NaN
982947             104-48 37 DR       104-48 37 DR          NaN
1868965      155 EAST 38 ST, 4B     155 EAST 38 ST           4B

Null ADDRESS_UNIT count: 1,618,395


In [15]:
sample_zip = combined[combined['ZIP CODE'] == 11214]
unique_bases = sample_zip['ADDRESS_BASE'].dropna().unique()

print(f"Unique base addresses in this ZIP: {len(unique_bases)}")

near_matches = []
for a, b in itertools.combinations(unique_bases, 2):
    score = fuzz.token_sort_ratio(a, b)
    if 80 <= score < 100:
        near_matches.append((a, b, score))

near_matches.sort(key=lambda x: -x[2])
for a, b, score in near_matches[:15]:
    print(f"{score:.0f}  {a!r:35} <-> {b!r}")

Unique base addresses in this ZIP: 7737
97  '2652 CROPSEY AVE'                  <-> '2652 CROPSEY AVE.'
97  '2015 CROPSEY AVE.'                 <-> '2015 CROPSEY AVE'
97  '2121 SHORE PKWY'                   <-> '2121 SHORE PKWAY'
97  '2545 WEST 17 ST'                   <-> '2545A WEST 17 ST'
97  '8853 BAY 16 ST'                    <-> '88-53 BAY 16 ST'
96  '87-35 BAY PKWY'                    <-> '8735 BAY PKWY'
96  '288 BAY 19 ST'                     <-> '288A BAY 19 ST'
96  '2116 BAY 28 ST'                    <-> '211 BAY 28 ST'
96  '2116 BAY 28 ST'                    <-> '116 BAY 28 ST'
96  '2118 BAY 28 ST'                    <-> '211 BAY 28 ST'
96  '2118 BAY 28 ST'                    <-> '118 BAY 28 ST'
96  '2120 BAY 28 ST'                    <-> '120 BAY 28 ST'
96  '127A BAY 49 ST'                    <-> '127 BAY 49 ST'
96  '222A BAY 48 ST'                    <-> '222 BAY 48 ST'
96  '129A BAY 44 ST'                    <-> '129 BAY 44 ST'


In [16]:
def split_house_number(base_addr):
    if pd.isna(base_addr):
        return None, None
    match = re.match(r'^(\d+[A-Z]?(?:-\d+)?)\s+(.*)$', base_addr)
    if match:
        return match.group(1), match.group(2)
    return None, base_addr  # no leading number found (e.g. "BROADWAY" alone)

test_cases = ['2116 BAY 28 ST', '211 BAY 28 ST', '2545 WEST 17 ST', '2545A WEST 17 ST',
              '87-35 BAY PKWY', '127A BAY 49 ST', '2652 CROPSEY AVE', '2652 CROPSEY AVE.']
for a in test_cases:
    hn, street = split_house_number(a)
    print(f"{a!r:25} -> house_number={hn!r:12} street={street!r}")

'2116 BAY 28 ST'          -> house_number='2116'       street='BAY 28 ST'
'211 BAY 28 ST'           -> house_number='211'        street='BAY 28 ST'
'2545 WEST 17 ST'         -> house_number='2545'       street='WEST 17 ST'
'2545A WEST 17 ST'        -> house_number='2545A'      street='WEST 17 ST'
'87-35 BAY PKWY'          -> house_number='87-35'      street='BAY PKWY'
'127A BAY 49 ST'          -> house_number='127A'       street='BAY 49 ST'
'2652 CROPSEY AVE'        -> house_number='2652'       street='CROPSEY AVE'
'2652 CROPSEY AVE.'       -> house_number='2652'       street='CROPSEY AVE.'


In [17]:
combined[['HOUSE_NUMBER', 'STREET_NAME']] = combined['ADDRESS_BASE'].apply(
    lambda x: pd.Series(split_house_number(x))
)

print(f"Nulls in HOUSE_NUMBER: {combined['HOUSE_NUMBER'].isnull().sum():,}")
print(f"Nulls in STREET_NAME: {combined['STREET_NAME'].isnull().sum():,}")
print(combined[['ADDRESS_BASE', 'HOUSE_NUMBER', 'STREET_NAME']].sample(10, random_state=2))

Nulls in HOUSE_NUMBER: 32,693
Nulls in STREET_NAME: 0
                  ADDRESS_BASE HOUSE_NUMBER     STREET_NAME
1941505         1325 FIFTH AVE         1325       FIFTH AVE
521947              1035 2 AVE         1035           2 AVE
1848433         563 EAST 55 ST          563      EAST 55 ST
2055538  105-50 CROSS BAY BLVD       105-50  CROSS BAY BLVD
229690         2012 JEROME AVE         2012      JEROME AVE
187834          231 EAST 96 ST          231      EAST 96 ST
887221        1816 CORNELIA ST         1816     CORNELIA ST
923578               INWOOD ST          NaN       INWOOD ST
827340           240-17 42 AVE       240-17          42 AVE
670613          151 WEST 17 ST          151      WEST 17 ST


In [18]:
def address_similarity_match(row_a, row_b, street_threshold=90):
    """
    Returns True if two records are a candidate match on address,
    combining exact-match fields (house number, unit) with fuzzy street name matching.
    """
    # House number: exact match required, skip comparison if either is null
    if pd.isna(row_a['HOUSE_NUMBER']) or pd.isna(row_b['HOUSE_NUMBER']):
        return None  # can't evaluate — not a match, not a non-match, just unknown
    if row_a['HOUSE_NUMBER'] != row_b['HOUSE_NUMBER']:
        return False

    # Unit: exact match required, but both-null counts as agreement (no unit = no unit)
    unit_a, unit_b = row_a['ADDRESS_UNIT'], row_b['ADDRESS_UNIT']
    if pd.isna(unit_a) and pd.isna(unit_b):
        pass  # both no unit — fine, continue
    elif pd.isna(unit_a) or pd.isna(unit_b):
        return None  # one has a unit, one doesn't — genuinely ambiguous, flag don't guess
    elif unit_a != unit_b:
        return False

    # Street name: fuzzy match allowed
    street_score = fuzz.token_sort_ratio(row_a['STREET_NAME'], row_b['STREET_NAME'])
    return street_score >= street_threshold

# Re-test on our known problem cases from Bay 28 St / Cropsey Ave
test_zip = combined[combined['ZIP CODE'] == 11214]
test_addrs = test_zip[test_zip['ADDRESS_BASE'].isin([
    '2116 BAY 28 ST', '211 BAY 28 ST', '116 BAY 28 ST',
    '2652 CROPSEY AVE', '2652 CROPSEY AVE.',
    '2545 WEST 17 ST', '2545A WEST 17 ST',
])].drop_duplicates(subset=['ADDRESS_BASE', 'ADDRESS_UNIT'])

for i, row_a in test_addrs.iterrows():
    for j, row_b in test_addrs.iterrows():
        if i >= j:
            continue
        result = address_similarity_match(row_a, row_b)
        print(f"{row_a['ADDRESS_BASE']!r:20} vs {row_b['ADDRESS_BASE']!r:20} -> {result}")

'2116 BAY 28 ST'     vs '211 BAY 28 ST'      -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '116 BAY 28 ST'      -> False
'2116 BAY 28 ST'     vs '2545 WEST 17 ST'    -> False
'2116 BAY 28 ST'     vs '2545A WEST 17 ST'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '2652 CROPSEY AVE'   -> False
'2116 BAY 28 ST'     vs '265

In [19]:
# Block on the fields that must match exactly — this collapses most of the
# work before any fuzzy string comparison happens
block_keys = combined.groupby(['ZIP CODE', 'HOUSE_NUMBER', 'ADDRESS_UNIT'], dropna=False)

print(f"Total (zip, house_number, unit) groups: {block_keys.ngroups:,}")

# Within each group, check how many DISTINCT street name spellings exist —
# most groups should have exactly 1 (no ambiguity at all)
street_variation = block_keys['STREET_NAME'].nunique()
print(street_variation.value_counts().sort_index())

Total (zip, house_number, unit) groups: 560,509
STREET_NAME
1      416530
2       58646
3       26014
4       16274
5       10953
        ...  
308         1
310         1
326         1
335         1
865         1
Name: count, Length: 139, dtype: int64


In [20]:
# Look at the biggest offending group directly
biggest_group_size = street_variation.idxmax()
print(f"Group key with most street variation: {biggest_group_size}")

sample_group = combined[
    (combined['ZIP CODE'] == biggest_group_size[0]) &
    (combined['HOUSE_NUMBER'].isna() if pd.isna(biggest_group_size[1]) else combined['HOUSE_NUMBER'] == biggest_group_size[1]) &
    (combined['ADDRESS_UNIT'].isna() if pd.isna(biggest_group_size[2]) else combined['ADDRESS_UNIT'] == biggest_group_size[2])
]
print(sample_group[['ADDRESS_BASE', 'HOUSE_NUMBER', 'STREET_NAME']].drop_duplicates().head(15))

Group key with most street variation: (np.float64(0.0), nan, nan)
               ADDRESS_BASE HOUSE_NUMBER           STREET_NAME
38974             BROOK AVE          NaN             BROOK AVE
38977           EAST 156 ST          NaN           EAST 156 ST
41735            REVERE AVE          NaN            REVERE AVE
55489   DR M L KING JR BLVD          NaN   DR M L KING JR BLVD
62045           LEGGETT AVE          NaN           LEGGETT AVE
67078  HARLEM RIVER TERRACE          NaN  HARLEM RIVER TERRACE
67118         OAK POINT AVE          NaN         OAK POINT AVE
71579           EAST 181 ST          NaN           EAST 181 ST
72844            TIMPSON PL          NaN            TIMPSON PL
81406            TERRACE PL          NaN            TERRACE PL
82017    EDWARD L GRANT HWY          NaN    EDWARD L GRANT HWY
85731             LYVERE ST          NaN             LYVERE ST
87459         TILLOTSON AVE          NaN         TILLOTSON AVE
87804           EAST 179 ST          NaN           E

In [21]:
has_house_number = combined[combined['HOUSE_NUMBER'].notna()]

block_keys2 = has_house_number.groupby(['ZIP CODE', 'HOUSE_NUMBER', 'ADDRESS_UNIT'], dropna=False)
street_variation2 = block_keys2['STREET_NAME'].nunique()

print(f"Total groups (house number present): {block_keys2.ngroups:,}")
print(street_variation2.value_counts().sort_index())

Total groups (house number present): 559,097
STREET_NAME
1      415351
2       58600
3       26006
4       16271
5       10951
        ...  
135         1
137         1
141         1
148         1
158         1
Name: count, Length: 119, dtype: int64


In [23]:
candidate_groups = has_house_number[has_house_number['ZIP CODE'] != 0]
grouped = candidate_groups.groupby(['ZIP CODE', 'HOUSE_NUMBER', 'ADDRESS_UNIT'], dropna=False)

matches = []
groups_with_multiple_streets = 0
pairs_checked = 0

for key, group in grouped:
    unique_streets = group['STREET_NAME'].unique()
    if len(unique_streets) < 2:
        continue  # only one spelling in this group, nothing to disambiguate
    groups_with_multiple_streets += 1
    for a, b in combinations(unique_streets, 2):
        pairs_checked += 1
        score = fuzz.token_sort_ratio(a, b)
        if score >= 90:
            matches.append((key, a, b, score))

print(f"Groups with 2+ distinct street spellings: {groups_with_multiple_streets:,}")
print(f"Total street-name pairs compared: {pairs_checked:,}")
print(f"Pairs matched (score >= 90): {len(matches):,}")

Groups with 2+ distinct street spellings: 143,624
Total street-name pairs compared: 2,773,288
Pairs matched (score >= 90): 91,802


In [24]:
import random
random.seed(42)
sample_matches = random.sample(matches, 20)

for key, a, b, score in sample_matches:
    print(f"{score:.0f}  {a!r:30} <-> {b!r:30}  (zip/house/unit: {key})")

96  'DITMARS BLVD'                 <-> 'DITMARS BLVD.'                 (zip/house/unit: (11370.0, '69-15', nan))
91  'WEST 112 ST'                  <-> 'WEST 118 ST'                   (zip/house/unit: (10026.0, '310', nan))
92  'WEST 23 ST APT. 16B'          <-> 'WEST 23 ST. APT. 14B'          (zip/house/unit: (10011.0, '520', nan))
90  'EAST 51 ST'                   <-> 'EAST 45 ST'                    (zip/house/unit: (11203.0, '375', nan))
91  'EAST 215 ST'                  <-> 'EAST 221 ST'                   (zip/house/unit: (10469.0, '932', nan))
91  'EAST 224 ST'                  <-> 'EAST 232 ST'                   (zip/house/unit: (10466.0, '920', nan))
91  'EAST 107 ST'                  <-> 'EAST 105 ST'                   (zip/house/unit: (10029.0, '160', nan))
90  'WEST 102 ST. APT. 11U'        <-> 'WEST 102 ST. APT. 14J'         (zip/house/unit: (10025.0, '235', nan))
92  'BEACH 145 ST'                 <-> 'BEACH 134 ST'                  (zip/house/unit: (11694.0, '140', nan))

In [25]:
def split_street_number(street):
    if pd.isna(street):
        return None, None
    match = re.search(r'\b(\d+)\b', street)
    if match:
        return match.group(1), street  # keep number separately, keep full string for the word-part comparison
    return None, street

# Test on exactly the false positives we just found
test_streets = ['WEST 112 ST', 'WEST 118 ST', 'EAST 51 ST', 'EAST 45 ST',
                 'DITMARS BLVD', 'DITMARS BLVD.', 'BRIGHTON 7 ST', 'BRIGHTON 1 ST']
for s in test_streets:
    num, _ = split_street_number(s)
    print(f"{s!r:20} -> street_number={num!r}")

'WEST 112 ST'        -> street_number='112'
'WEST 118 ST'        -> street_number='118'
'EAST 51 ST'         -> street_number='51'
'EAST 45 ST'         -> street_number='45'
'DITMARS BLVD'       -> street_number=None
'DITMARS BLVD.'      -> street_number=None
'BRIGHTON 7 ST'      -> street_number='7'
'BRIGHTON 1 ST'      -> street_number='1'


In [26]:
def street_similarity_match(street_a, street_b, word_threshold=90):
    """
    Returns True only if: any numeric token in the street name matches exactly,
    AND the non-numeric portion is similar enough to be a spelling/formatting variant.
    """
    num_a, _ = split_street_number(street_a)
    num_b, _ = split_street_number(street_b)

    # If either has a number, both must have the SAME number
    if num_a is not None or num_b is not None:
        if num_a != num_b:
            return False

    # Compare the word-level content (numbers included is fine now, since we've
    # already enforced they match or are both absent)
    score = fuzz.token_sort_ratio(street_a, street_b)
    return score >= word_threshold

# Re-verify against our known false positives and true positives
test_pairs = [
    ('WEST 112 ST', 'WEST 118 ST'),      # should be False now
    ('EAST 51 ST', 'EAST 45 ST'),        # should be False now
    ('BRIGHTON 7 ST', 'BRIGHTON 1 ST'),  # should be False now
    ('DITMARS BLVD', 'DITMARS BLVD.'),   # should stay True
]
for a, b in test_pairs:
    print(f"{a!r:20} <-> {b!r:20} -> {street_similarity_match(a, b)}")

'WEST 112 ST'        <-> 'WEST 118 ST'        -> False
'EAST 51 ST'         <-> 'EAST 45 ST'         -> False
'BRIGHTON 7 ST'      <-> 'BRIGHTON 1 ST'      -> False
'DITMARS BLVD'       <-> 'DITMARS BLVD.'      -> True


In [27]:
matches_v2 = []
for key, group in grouped:
    unique_streets = group['STREET_NAME'].unique()
    if len(unique_streets) < 2:
        continue
    for a, b in combinations(unique_streets, 2):
        if street_similarity_match(a, b):
            matches_v2.append((key, a, b))

print(f"Pairs matched (corrected logic): {len(matches_v2):,}")

Pairs matched (corrected logic): 14,885


In [28]:
random.seed(7)
sample_v2 = random.sample(matches_v2, 20)

for key, a, b in sample_v2:
    print(f"{a!r:30} <-> {b!r:30}  (zip/house/unit: {key})")

'WEST 145 ST APT. 712'         <-> 'WEST 145 ST APT. 301'          (zip/house/unit: (10039.0, '330', nan))
'RIVERSIDE DR APT. 7HE'        <-> 'RIVERSIDE DR APT. 2VE'         (zip/house/unit: (10023.0, '11', nan))
'W FINGERBOARD RD'             <-> 'W. FINGERBOARD RD'             (zip/house/unit: (10304.0, '980', nan))
'ST PAULS PL'                  <-> "ST. PAUL'S PL"                 (zip/house/unit: (11226.0, '39', nan))
'WEST 23 ST. APT. 8E'          <-> 'WEST 23 ST. APT. 8A'           (zip/house/unit: (10011.0, '520', nan))
'EAST 40 ST. APT. 18G'         <-> 'EAST 40 ST. APT. 16C'          (zip/house/unit: (10016.0, '300', nan))
'108 ST'                       <-> '108 ST.'                       (zip/house/unit: (11375.0, '67-38', nan))
'CLINTON AV'                   <-> 'CLINTON AVE'                   (zip/house/unit: (11205.0, '210', nan))
'EAST 46 ST APT. 12K'          <-> 'EAST 46 ST. APT. 12E'          (zip/house/unit: (10017.0, '310', nan))
'FORT WASHINGTON AVEN'         <-> 'F

In [29]:
test_addrs = [
    'WEST 145 ST APT. 712', 'RIVERSIDE DR APT. 7HE', 'EAST 40 ST. APT. 18G',
    'MADISON AVE. APT. 2D', 'FREDERICK DOUGLASS BLVD APT. 4D',
]
for a in test_addrs:
    print(f"{a!r:35} -> {split_address(a)}")

'WEST 145 ST APT. 712'              -> ('WEST 145 ST APT. 712', None)
'RIVERSIDE DR APT. 7HE'             -> ('RIVERSIDE DR APT. 7HE', None)
'EAST 40 ST. APT. 18G'              -> ('EAST 40 ST. APT. 18G', None)
'MADISON AVE. APT. 2D'              -> ('MADISON AVE. APT. 2D', None)
'FREDERICK DOUGLASS BLVD APT. 4D'   -> ('FREDERICK DOUGLASS BLVD APT. 4D', None)


In [30]:
def split_address(addr):
    if pd.isna(addr):
        return None, None
    # First try comma-based split (original logic)
    if ',' in addr:
        base, unit = addr.split(',', 1)
        return base.strip(), unit.strip()
    # NEW: try splitting on APT/APARTMENT/UNIT/# even without a comma
    match = re.search(r'\b(APT\.?|APARTMENT|UNIT|#)\s*(\S+)', addr)
    if match:
        base = addr[:match.start()].strip()
        unit = match.group(0).strip()
        return base, unit
    return addr.strip(), None

# Re-test on the exact cases that just failed
test_addrs = [
    'WEST 145 ST APT. 712', 'RIVERSIDE DR APT. 7HE', 'EAST 40 ST. APT. 18G',
    'MADISON AVE. APT. 2D', 'FREDERICK DOUGLASS BLVD APT. 4D',
    '2630 CROPSEY AVE, 1G',  # comma case, must still work
    '73 ST',                 # no unit at all, must still work
]
for a in test_addrs:
    print(f"{a!r:35} -> {split_address(a)}")

'WEST 145 ST APT. 712'              -> ('WEST 145 ST', 'APT. 712')
'RIVERSIDE DR APT. 7HE'             -> ('RIVERSIDE DR', 'APT. 7HE')
'EAST 40 ST. APT. 18G'              -> ('EAST 40 ST.', 'APT. 18G')
'MADISON AVE. APT. 2D'              -> ('MADISON AVE.', 'APT. 2D')
'FREDERICK DOUGLASS BLVD APT. 4D'   -> ('FREDERICK DOUGLASS BLVD', 'APT. 4D')
'2630 CROPSEY AVE, 1G'              -> ('2630 CROPSEY AVE', '1G')
'73 ST'                             -> ('73 ST', None)


In [31]:
# Full re-derivation, in the correct order, on the full dataset
combined[['ADDRESS_BASE', 'ADDRESS_UNIT']] = combined['ADDRESS_NORM'].apply(
    lambda x: pd.Series(split_address(x))
)
combined['ADDRESS_UNIT'] = combined['ADDRESS_UNIT'].apply(normalize_unit)

combined[['HOUSE_NUMBER', 'STREET_NAME']] = combined['ADDRESS_BASE'].apply(
    lambda x: pd.Series(split_house_number(x))
)

print(f"Null ADDRESS_UNIT: {combined['ADDRESS_UNIT'].isnull().sum():,} (was 1,618,395 before fix)")
print(f"Null HOUSE_NUMBER: {combined['HOUSE_NUMBER'].isnull().sum():,}")

combined.to_csv("../data/processed/nyc_sales_2003_2025_combined.csv", index=False)
print("Re-saved with corrected address parsing.")

Null ADDRESS_UNIT: 1,605,395 (was 1,618,395 before fix)
Null HOUSE_NUMBER: 33,788
Re-saved with corrected address parsing.


In [32]:
has_house_number = combined[combined['HOUSE_NUMBER'].notna()]
candidate_groups = has_house_number[has_house_number['ZIP CODE'] != 0]
grouped = candidate_groups.groupby(['ZIP CODE', 'HOUSE_NUMBER', 'ADDRESS_UNIT'], dropna=False)

matches_v3 = []
for key, group in grouped:
    unique_streets = group['STREET_NAME'].unique()
    if len(unique_streets) < 2:
        continue
    for a, b in combinations(unique_streets, 2):
        if street_similarity_match(a, b):
            matches_v3.append((key, a, b))

print(f"Total groups: {grouped.ngroups:,}")
print(f"Pairs matched (v3, corrected unit parsing): {len(matches_v3):,}")

Total groups: 563,852
Pairs matched (v3, corrected unit parsing): 10,838


In [33]:
random.seed(13)
sample_v3 = random.sample(matches_v3, 20)

for key, a, b in sample_v3:
    print(f"{a!r:30} <-> {b!r:30}  (zip/house/unit: {key})")

'TENBROECK AVE'                <-> 'TENBROECK AVE.'                (zip/house/unit: (10469.0, '2734', nan))
'DEKALB AVE'                   <-> 'DEKALB AVE.'                   (zip/house/unit: (11201.0, '124', nan))
"ST ANDREW'S RD"               <-> 'ST ANDREWS RD'                 (zip/house/unit: (10306.0, '429', nan))
'DANIELS ST'                   <-> 'DANIEL ST'                     (zip/house/unit: (11435.0, '84-65', nan))
'VAN CORTLANDT PARK S'         <-> 'VAN CORTLANDT PARK SOUTH'      (zip/house/unit: (10463.0, '180', '1C'))
'WEST 181 ST.'                 <-> 'WEST 181 ST'                   (zip/house/unit: (10033.0, '870', '68'))
'ST RAYMOND AVE'               <-> 'ST. RAYMOND AVE'               (zip/house/unit: (10462.0, '2065', nan))
'POLHEMAS AVE'                 <-> 'POLHEMUS AVE'                  (zip/house/unit: (11433.0, '174-17', nan))
'AMBOY RD'                     <-> 'AMBOY RD.'                     (zip/house/unit: (10308.0, '3747', nan))
'EAST 86 ST'               

In [34]:
# Union-Find: lets us merge "DEKALB AVE" and "DEKALB AVE." into one group,
# then merge that group with anything else that matches either of them.
parent = {}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]  # path compression, keeps it fast
        x = parent[x]
    return x

def union(x, y):
    parent.setdefault(x, x)
    parent.setdefault(y, y)
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[rx] = ry

# Tiny worked example before we touch real data
union(('demo', 'DEKALB AVE'), ('demo', 'DEKALB AVE.'))
union(('demo', 'DEKALB AVE.'), ('demo', 'DEKALB AV'))  # transitively joins all three

print(find(('demo', 'DEKALB AVE')))
print(find(('demo', 'DEKALB AVE.')))
print(find(('demo', 'DEKALB AV')))

('demo', 'DEKALB AV')
('demo', 'DEKALB AV')
('demo', 'DEKALB AV')


In [35]:
parent = {}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    parent.setdefault(x, x)
    parent.setdefault(y, y)
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[rx] = ry

# Build unions from every matched pair, scoped to its blocking key
# so "DEKALB AVE" in one zip never merges with "DEKALB AVE" in a different zip/house/unit
for key, a, b in matches_v3:
    node_a = (key, a)
    node_b = (key, b)
    union(node_a, node_b)

# Assign every row in the dataset its canonical (merged) street-group identity
def get_canonical(row):
    key = (row['ZIP CODE'], row['HOUSE_NUMBER'], row['ADDRESS_UNIT'])
    node = (key, row['STREET_NAME'])
    if node in parent:
        return find(node)
    return node  # ungrouped, its own singleton identity

combined['MATCH_GROUP'] = combined.apply(get_canonical, axis=1)

print(f"Total rows: {len(combined):,}")
print(f"Distinct MATCH_GROUP values: {combined['MATCH_GROUP'].nunique():,}")

Total rows: 2,070,484
Distinct MATCH_GROUP values: 1,049,913


In [36]:
has_bbl = combined[combined['BBL'].notna()].copy()
print(f"Rows with BBL (2016-2025 subset): {len(has_bbl):,}")

Rows with BBL (2016-2025 subset): 816,079


In [38]:
print("Largest MATCH_GROUP sizes:")
print(has_bbl.groupby('MATCH_GROUP').size().sort_values(ascending=False).head(10))
print("\nLargest BBL group sizes:")
print(has_bbl.groupby('BBL').size().sort_values(ascending=False).head(10))

Largest MATCH_GROUP sizes:
MATCH_GROUP
((10019.0, 102, nan), WEST 57 ST.)       551
((11101.0, 23-15, nan), 44 DR)           347
((10023.0, 15, nan), WEST 61 ST)         287
((10128.0, 200, nan), EAST 94 ST)        286
((10025.0, 100, nan), WEST 93 ST)        284
((10016.0, 330, nan), EAST 38 ST)        277
((11354.0, 131-03, nan), 40 RD)          273
((11354.0, 131-05, nan), 40 RD)          270
((11238.0, 550, nan), VANDERBILD AVE)    268
((11224.0, 501, nan), SURF AVE.)         261
dtype: int64

Largest BBL group sizes:
BBL
0             1421
2039447501    1403
4084890001    1356
1002487501     913
4004377502     856
4058937501     759
3072797501     747
2039437501     744
1000940001     713
1010090037     582
dtype: int64


In [39]:
has_bbl_clean = has_bbl[has_bbl['BBL'] != 0].copy()
print(f"Rows before BBL=0 filter: {len(has_bbl):,}")
print(f"Rows after BBL=0 filter: {len(has_bbl_clean):,}")

Rows before BBL=0 filter: 816,079
Rows after BBL=0 filter: 814,658


In [40]:
def get_pairs_from_groups(df, group_col):
    pairs = set()
    for _, group in df.groupby(group_col):
        idxs = sorted(group.index.tolist())
        if len(idxs) < 2:
            continue
        for a, b in combinations(idxs, 2):
            pairs.add((a, b))
    return pairs

predicted_pairs = get_pairs_from_groups(has_bbl_clean, 'MATCH_GROUP')
actual_pairs = get_pairs_from_groups(has_bbl_clean, 'BBL')

print(f"Predicted pairs (address-matched): {len(predicted_pairs):,}")
print(f"Actual pairs (same BBL): {len(actual_pairs):,}")

Predicted pairs (address-matched): 2,957,887
Actual pairs (same BBL): 19,582,970


In [41]:
true_positives = predicted_pairs & actual_pairs

precision = len(true_positives) / len(predicted_pairs)
recall = len(true_positives) / len(actual_pairs)

print(f"True positives (correctly matched pairs): {len(true_positives):,}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

True positives (correctly matched pairs): 2,946,763
Precision: 0.9962
Recall: 0.1505


In [42]:
# For BBL pairs we MISSED, check: do the two rows actually have the SAME
# ADDRESS_UNIT (meaning they really are the same physical unit, a genuine miss)
# or DIFFERENT units (meaning same building, different apartments - not a real miss)?

missed_pairs = actual_pairs - predicted_pairs
print(f"Missed pairs: {len(missed_pairs):,}")

# Sample 1000 missed pairs and check their units
sample_missed = random.sample(list(missed_pairs), 1000)

same_unit_count = 0
diff_unit_count = 0
both_null_unit_count = 0

for a, b in sample_missed:
    unit_a = has_bbl_clean.loc[a, 'ADDRESS_UNIT']
    unit_b = has_bbl_clean.loc[b, 'ADDRESS_UNIT']
    if pd.isna(unit_a) and pd.isna(unit_b):
        both_null_unit_count += 1
    elif unit_a == unit_b:
        same_unit_count += 1
    else:
        diff_unit_count += 1

print(f"Missed pairs with SAME unit (genuine misses): {same_unit_count}")
print(f"Missed pairs with DIFFERENT units (condo building artifact): {diff_unit_count}")
print(f"Missed pairs with BOTH units null (ambiguous, could be genuine miss or single-family artifact): {both_null_unit_count}")

Missed pairs: 16,636,207
Missed pairs with SAME unit (genuine misses): 2
Missed pairs with DIFFERENT units (condo building artifact): 940
Missed pairs with BOTH units null (ambiguous, could be genuine miss or single-family artifact): 58


In [43]:
sample_both_null = [(a, b) for a, b in sample_missed 
                     if pd.isna(has_bbl_clean.loc[a, 'ADDRESS_UNIT']) 
                     and pd.isna(has_bbl_clean.loc[b, 'ADDRESS_UNIT'])][:10]

for a, b in sample_both_null:
    row_a = has_bbl_clean.loc[a]
    row_b = has_bbl_clean.loc[b]
    print(f"A: {row_a['ADDRESS_BASE']!r} | {row_a['SALE DATE'].date()} | ${row_a['SALE PRICE']:,.0f}")
    print(f"B: {row_b['ADDRESS_BASE']!r} | {row_b['SALE DATE'].date()} | ${row_b['SALE PRICE']:,.0f}")
    print(f"   BBL: {row_a['BBL']} == {row_b['BBL']}")
    print()

A: '229-07 69 AVE' | 2016-07-21 | $450,000
B: '6777A CLOVERDALE LN' | 2018-04-02 | $0
   BBL: 4076747501 == 4076747501

A: '1 METROPOLITAN OVAL' | 2020-05-22 | $242,346
B: '1725 PURDY ST' | 2018-03-08 | $113,568
   BBL: 2039447501 == 2039447501

A: '33 JOSEPH LN' | 2021-07-06 | $599,000
B: '8 JOSEPH LN' | 2018-10-16 | $0
   BBL: 5030247502 == 5030247502

A: '27010 GRAND CENTRAL PKWY' | 2020-09-30 | $360,000
B: '27110 GRAND CENTRAL PKWY' | 2020-10-05 | $683,981
   BBL: 4084890001 == 4084890001

A: '1940 EAST TREMONT AVE' | 2016-07-27 | $103,000
B: '2120 EAST TREMONT AVE' | 2017-02-09 | $175,000
   BBL: 2039447501 == 2039447501

A: '1563 METROPOLITAN AVE' | 2021-08-27 | $270,000
B: '1541 METROPOLITAN AVE' | 2018-01-25 | $190,000
   BBL: 2039447501 == 2039447501

A: '50-09 2 ST' | 2020-07-29 | $880,786
B: '5009 2 ST' | 2016-01-28 | $860,000
   BBL: 4000167501 == 4000167501

A: '1505 METROPOLITAN AVE' | 2020-10-13 | $0
B: '1735 PURDY ST' | 2016-04-28 | $122,000
   BBL: 2039447501 == 203944

In [1]:
combined.to_csv("../data/processed/nyc_sales_2003_2025_matched.csv", index=False)
print("Saved:", combined.shape)

NameError: name 'combined' is not defined

In [3]:
import pandas as pd
import re
import random
from itertools import combinations
from rapidfuzz import fuzz

text_cols = ["ADDRESS", "ADDRESS_NORM", "ADDRESS_BASE", "ADDRESS_UNIT",
             "HOUSE_NUMBER", "STREET_NAME", "APARTMENT NUMBER"]

combined = pd.read_csv(
    "../data/processed/nyc_sales_2003_2025_combined.csv",
    dtype={c: "str" for c in text_cols},
    low_memory=False,
)
combined["SALE DATE"] = pd.to_datetime(combined["SALE DATE"])
combined["BBL"] = combined["BBL"].astype("Int64")

print(combined.shape)
print([c for c in text_cols if c in combined.columns])
print("Has MATCH_GROUP:", "MATCH_GROUP" in combined.columns)

(2070484, 34)
['ADDRESS', 'ADDRESS_NORM', 'ADDRESS_BASE', 'ADDRESS_UNIT', 'HOUSE_NUMBER', 'STREET_NAME', 'APARTMENT NUMBER']
Has MATCH_GROUP: False


In [4]:
print(combined['ADDRESS_BASE'].str.contains('#', na=False).sum())

226


In [6]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

from itertools import combinations
from src.matching import street_similarity_match, UnionFind

NO_UNIT = "<NONE>"

cand = combined[combined['HOUSE_NUMBER'].notna() & (combined['ZIP CODE'] != 0)].copy()
cand['UNIT_KEY'] = cand['ADDRESS_UNIT'].fillna(NO_UNIT)
cand['ZIP_KEY'] = cand['ZIP CODE'].fillna(-1)

grouped = cand.groupby(['ZIP_KEY', 'HOUSE_NUMBER', 'UNIT_KEY'])
print(f"Total groups: {grouped.ngroups:,}")      # earlier run: 563,852

uf = UnionFind()
pairs_matched = 0
for key, g in grouped:
    streets = g['STREET_NAME'].unique()
    if len(streets) < 2:
        continue
    for a, b in combinations(streets, 2):
        if street_similarity_match(a, b):
            uf.union((key, a), (key, b))
            pairs_matched += 1

print(f"Pairs matched: {pairs_matched:,}")       # earlier run: 10,838

Total groups: 563,267
Pairs matched: 10,840


In [7]:
print("Null ADDRESS_UNIT now :", combined['ADDRESS_UNIT'].isna().sum(), "  (earlier: 1,605,395)")
print("Null HOUSE_NUMBER now :", combined['HOUSE_NUMBER'].isna().sum(), "  (earlier: 33,788)")
print("Null STREET_NAME now  :", combined['STREET_NAME'].isna().sum(), "  (earlier: 0)")

raw = pd.read_csv(
    "../data/processed/nyc_sales_2003_2025_combined.csv",
    usecols=['ADDRESS_UNIT', 'HOUSE_NUMBER'],
    dtype=str, keep_default_na=False, na_values=[''],
)

for col in ['ADDRESS_UNIT', 'HOUSE_NUMBER']:
    changed = combined[col].isna() & raw[col].notna()
    print(f"\n{col}: values silently turned into NaN by the reload: {changed.sum():,}")
    print(raw.loc[changed, col].value_counts().head(10))

Null ADDRESS_UNIT now : 1606200   (earlier: 1,605,395)
Null HOUSE_NUMBER now : 33788   (earlier: 33,788)
Null STREET_NAME now  : 39   (earlier: 0)

ADDRESS_UNIT: values silently turned into NaN by the reload: 805
ADDRESS_UNIT
N/A    800
NA       5
Name: count, dtype: int64

HOUSE_NUMBER: values silently turned into NaN by the reload: 0
Series([], Name: count, dtype: int64)


In [8]:
text_cols = ["ADDRESS", "ADDRESS_NORM", "ADDRESS_BASE", "ADDRESS_UNIT",
             "HOUSE_NUMBER", "STREET_NAME", "APARTMENT NUMBER"]

combined = pd.read_csv(
    "../data/processed/nyc_sales_2003_2025_combined.csv",
    dtype={c: "str" for c in text_cols},
    keep_default_na=False,
    na_values=[''],          # only truly empty cells count as missing
    low_memory=False,
)
combined["SALE DATE"] = pd.to_datetime(combined["SALE DATE"])
combined["BBL"] = combined["BBL"].astype("Int64")

print(combined.shape)
print("Null ADDRESS_UNIT :", combined['ADDRESS_UNIT'].isna().sum(), "(target: 1,605,395)")
print("Null HOUSE_NUMBER :", combined['HOUSE_NUMBER'].isna().sum(), "(target: 33,788)")
print("Null STREET_NAME  :", combined['STREET_NAME'].isna().sum(), "(target: 0)")

(2070484, 34)
Null ADDRESS_UNIT : 1605395 (target: 1,605,395)
Null HOUSE_NUMBER : 33788 (target: 33,788)
Null STREET_NAME  : 37 (target: 0)


In [9]:
bad = combined[combined['STREET_NAME'].isna()]
print(bad[['ADDRESS', 'ADDRESS_NORM', 'ADDRESS_BASE', 'HOUSE_NUMBER']].head(8).to_string())
print("\nRows with null STREET_NAME:", len(bad))
print("...of which HOUSE_NUMBER is also null:", bad['HOUSE_NUMBER'].isna().sum())

                                          ADDRESS               ADDRESS_NORM ADDRESS_BASE HOUSE_NUMBER
62979                                                                    NaN          NaN          NaN
62980                                                                    NaN          NaN          NaN
63013                                                                    NaN          NaN          NaN
156505  UNIT#11 M 135 OCEAN PKWAY                  UNIT#11 M 135 OCEAN PKWAY          NaN          NaN
453325  APT#22A                                                      APT#22A          NaN          NaN
726984  UNITED NATIONS PLAZA                            UNITED NATIONS PLAZA          NaN          NaN
726985  UNITED NATIONS PLAZA                            UNITED NATIONS PLAZA          NaN          NaN
726988  UNITED NATIONS PLAZA                            UNITED NATIONS PLAZA          NaN          NaN

Rows with null STREET_NAME: 37
...of which HOUSE_NUMBER is also null: 37

In [10]:
no_comma = ~combined['ADDRESS_NORM'].str.contains(',', na=False)
word = combined['ADDRESS_NORM'].str.extract(r'\b((?:UNIT|APT)[A-Z]+)', expand=False)
affected = no_comma & word.notna()

print("Rows affected:", affected.sum())
print(word[affected].value_counts().head(15))

Rows affected: 1117
ADDRESS_NORM
UNITED    1117
Name: count, dtype: int64


In [11]:
from src.matching import split_address, normalize_unit

for a in ['UNITED NATIONS PLAZA', '860 UNITED NATIONS PLAZA', 'WEST 145 ST APT. 712',
          'MADISON AVE. APT. 2D', '123 MAIN ST #4B', 'APT#22A',
          '2630 CROPSEY AVE, 1G', '73 ST']:
    print(f"{a!r:30} -> {split_address(a)}")

print()
for u in ['1G', '11G', 'APT 8E', 'APT 8', '#4B', 'UNIT 12', 'APT#22A', 'UNITED', '']:
    print(f"{u!r:12} -> {normalize_unit(u)!r}")

'UNITED NATIONS PLAZA'         -> ('UNITED NATIONS PLAZA', None)
'860 UNITED NATIONS PLAZA'     -> ('860 UNITED NATIONS PLAZA', None)
'WEST 145 ST APT. 712'         -> ('WEST 145 ST', 'APT. 712')
'MADISON AVE. APT. 2D'         -> ('MADISON AVE.', 'APT. 2D')
'123 MAIN ST #4B'              -> ('123 MAIN ST', '#4B')
'APT#22A'                      -> ('', 'APT#22A')
'2630 CROPSEY AVE, 1G'         -> ('2630 CROPSEY AVE', '1G')
'73 ST'                        -> ('73 ST', None)

'1G'         -> '1G'
'11G'        -> '11G'
'APT 8E'     -> '8E'
'APT 8'      -> '8'
'#4B'        -> '4B'
'UNIT 12'    -> '12'
'APT#22A'    -> '22A'
'UNITED'     -> 'UNITED'
''           -> None


In [14]:
from src.matching import split_address, normalize_unit, split_house_number

idx = combined.index

parts = combined['ADDRESS_NORM'].apply(split_address)
new_base   = pd.Series([p[0] or None for p in parts], index=idx)
new_unit   = pd.Series([normalize_unit(p[1]) for p in parts], index=idx)
hn         = [split_house_number(b) for b in new_base]
new_house  = pd.Series([h[0] for h in hn], index=idx)
new_street = pd.Series([h[1] or None for h in hn], index=idx)

def differs(old, new):
    old, new = old.astype(object), new.astype(object)
    return ~((old.isna() & new.isna()) | (old == new))

d_base   = differs(combined['ADDRESS_BASE'],  new_base)
d_unit   = differs(combined['ADDRESS_UNIT'],  new_unit)
d_house  = differs(combined['HOUSE_NUMBER'],  new_house)
d_street = differs(combined['STREET_NAME'],   new_street)
any_diff = d_base | d_unit | d_house | d_street

expected   = combined['ADDRESS_NORM'].str.contains(r'\bUNITED\b|#', na=False)
unexpected = any_diff & ~expected

print("Rows changed in ADDRESS_BASE  :", d_base.sum())
print("Rows changed in ADDRESS_UNIT  :", d_unit.sum())
print("Rows changed in HOUSE_NUMBER  :", d_house.sum())
print("Rows changed in STREET_NAME   :", d_street.sum())
print("Total rows with any change    :", any_diff.sum())
print("...of which expected (UNITED / '#'):", (any_diff & expected).sum())
print("...of which UNEXPECTED             :", unexpected.sum())

print("\n--- 6 expected changes (eyeball these) ---")
show = any_diff & expected
print(pd.DataFrame({
    'ADDRESS_NORM': combined.loc[show, 'ADDRESS_NORM'],
    'old_unit': combined.loc[show, 'ADDRESS_UNIT'],
    'new_unit': new_unit[show],
    'old_house': combined.loc[show, 'HOUSE_NUMBER'],
    'new_house': new_house[show],
    'new_street': new_street[show],
}).sample(6, random_state=1).to_string())

print("\n--- UNEXPECTED changes (should be empty) ---")
print(pd.DataFrame({
    'ADDRESS_NORM': combined.loc[unexpected, 'ADDRESS_NORM'],
    'old_base': combined.loc[unexpected, 'ADDRESS_BASE'],
    'new_base': new_base[unexpected],
    'old_unit': combined.loc[unexpected, 'ADDRESS_UNIT'],
    'new_unit': new_unit[unexpected],
}).head(10).to_string())

Rows changed in ADDRESS_BASE  : 1340
Rows changed in ADDRESS_UNIT  : 5834
Rows changed in HOUSE_NUMBER  : 1092
Rows changed in STREET_NAME   : 1340
Total rows with any change    : 5834
...of which expected (UNITED / '#'): 5831
...of which UNEXPECTED             : 3

--- 6 expected changes (eyeball these) ---
                   ADDRESS_NORM old_unit new_unit old_house new_house    new_street
744569       905 166 ST, APT#9A      #9A       9A       905       905        166 ST
456206  55 EAST END AVE, APT#9E      #9E       9E        55        55  EAST END AVE
446646  687 WEST 204 ST, APT#5G      #5G       5G       687       687   WEST 204 ST
453463   205 WEST 63 ST, APT#2F      #2F       2F       205       205    WEST 63 ST
100485        302 96 ST, APT#1W      #1W       1W       302       302         96 ST
779840       3950 52 ST, APT#3E      #3E       3E      3950      3950         52 ST

--- UNEXPECTED changes (should be empty) ---
                                ADDRESS_NORM        old_

In [13]:
from src.matching import normalize_unit
for u in ['UNITC', 'APTA', 'APTGF', 'UNITB', 'APTS 3F & 3G', 'UNITED', 'APT#22A', 'APT 8E', '#4B', '1G', '']:
    print(f"{u!r:16} -> {normalize_unit(u)!r}")

'UNITC'          -> 'C'
'APTA'           -> 'A'
'APTGF'          -> 'GF'
'UNITB'          -> 'B'
'APTS 3F & 3G'   -> '3F & 3G'
'UNITED'         -> 'UNITED'
'APT#22A'        -> '22A'
'APT 8E'         -> '8E'
'#4B'            -> '4B'
'1G'             -> '1G'
''               -> None


In [15]:
combined['ADDRESS_BASE'] = new_base
combined['ADDRESS_UNIT'] = new_unit
combined['HOUSE_NUMBER'] = new_house
combined['STREET_NAME']  = new_street

print("Null ADDRESS_UNIT :", combined['ADDRESS_UNIT'].isna().sum())
print("Null HOUSE_NUMBER :", combined['HOUSE_NUMBER'].isna().sum())
print("Null STREET_NAME  :", combined['STREET_NAME'].isna().sum())

Null ADDRESS_UNIT : 1606305
Null HOUSE_NUMBER : 32698
Null STREET_NAME  : 11


In [16]:
%pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 35.6 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
path = "../data/processed/nyc_sales_2003_2025_parsed.parquet"
combined.to_parquet(path, index=False)

chk = pd.read_parquet(path)
print("Shape:", chk.shape, "(expected:", combined.shape, ")")
for c in ['ADDRESS_BASE', 'ADDRESS_UNIT', 'HOUSE_NUMBER', 'STREET_NAME', 'ADDRESS_NORM']:
    same = (chk[c].astype(object).fillna('~') == combined[c].astype(object).fillna('~')).all()
    print(f"{c:14} identical after round-trip: {same}")

ArrowKeyError: No type extension with name arrow.py_extension_type found

In [18]:
import sys
sys.path.append("..")
import pandas as pd

from src.matching import split_address, normalize_unit, split_house_number

text_cols = ["ADDRESS", "ADDRESS_NORM", "ADDRESS_BASE", "ADDRESS_UNIT",
             "HOUSE_NUMBER", "STREET_NAME", "APARTMENT NUMBER"]

combined = pd.read_csv(
    "../data/processed/nyc_sales_2003_2025_combined.csv",
    dtype={c: "str" for c in text_cols},
    keep_default_na=False, na_values=[''],
    low_memory=False,
)
combined["SALE DATE"] = pd.to_datetime(combined["SALE DATE"])
combined["BBL"] = combined["BBL"].astype("Int64")

idx = combined.index
parts = combined['ADDRESS_NORM'].apply(split_address)
combined['ADDRESS_BASE'] = pd.Series([p[0] or None for p in parts], index=idx)
combined['ADDRESS_UNIT'] = pd.Series([normalize_unit(p[1]) for p in parts], index=idx)
hn = [split_house_number(b) for b in combined['ADDRESS_BASE']]
combined['HOUSE_NUMBER'] = pd.Series([h[0] for h in hn], index=idx)
combined['STREET_NAME']  = pd.Series([h[1] or None for h in hn], index=idx)

print("Shape:", combined.shape)
print("Null ADDRESS_UNIT :", combined['ADDRESS_UNIT'].isna().sum(), "(expect 1,606,305)")
print("Null HOUSE_NUMBER :", combined['HOUSE_NUMBER'].isna().sum(), "(expect 32,698)")
print("Null STREET_NAME  :", combined['STREET_NAME'].isna().sum(), "(expect 11)")

Shape: (2070484, 34)
Null ADDRESS_UNIT : 1606305 (expect 1,606,305)
Null HOUSE_NUMBER : 32698 (expect 32,698)
Null STREET_NAME  : 11 (expect 11)


In [20]:
import pandas as pd, pyarrow
print(pd.__version__, pyarrow.__version__, pyarrow.__file__)

3.0.5 26.0.0 /Users/bhavingaikwad/Documents/DS_project/nyc-hpi-engine/venv/lib/python3.13/site-packages/pyarrow/__init__.py


In [21]:
import pandas as pd
pd.DataFrame({"a": ["x", None], "b": [1, 2]}).to_parquet("/tmp/tiny_test.parquet", index=False)
print(pd.read_parquet("/tmp/tiny_test.parquet"))

ArrowKeyError: A type extension with name pandas.period already defined

In [22]:
combined.to_pickle("../data/processed/nyc_sales_2003_2025_parsed.pkl")
chk = pd.read_pickle("../data/processed/nyc_sales_2003_2025_parsed.pkl")
print(chk.shape, chk['ADDRESS_UNIT'].isna().sum(), "(expect 1,606,305)")

(2070484, 34) 1606305 (expect 1,606,305)


In [23]:
from itertools import combinations
from src.matching import street_similarity_match, UnionFind

NO_UNIT = "<NONE>"

cand = combined[
    combined['HOUSE_NUMBER'].notna()
    & combined['STREET_NAME'].notna()
    & (combined['ZIP CODE'] != 0)
].copy()
cand['UNIT_KEY'] = cand['ADDRESS_UNIT'].fillna(NO_UNIT)
cand['ZIP_KEY']  = cand['ZIP CODE'].fillna(-1).astype(int)

grouped = cand.groupby(['ZIP_KEY', 'HOUSE_NUMBER', 'UNIT_KEY'])
print(f"Candidate rows: {len(cand):,}   Blocking groups: {grouped.ngroups:,}")

uf = UnionFind()
pairs_matched = 0
for key, g in grouped:
    streets = g['STREET_NAME'].unique()
    if len(streets) < 2:
        continue
    for a, b in combinations(streets, 2):
        if street_similarity_match(a, b):
            uf.union((key, a), (key, b))
            pairs_matched += 1
print(f"Street-name pairs matched: {pairs_matched:,}")

def node_str(node):
    (z, h, u), s = node
    return f"{z}|{h}|{u}|{s}"

mapping = {node_str(n): node_str(uf.find(n)) for n in list(uf.parent)}
print(f"Street spellings merged into another spelling: {len(mapping) - len(set(mapping.values())):,}")

cand['NODE'] = (cand['ZIP_KEY'].astype(str) + '|' + cand['HOUSE_NUMBER'] + '|'
                + cand['UNIT_KEY'] + '|' + cand['STREET_NAME'])
cand['MATCH_GROUP'] = cand['NODE'].map(mapping).fillna(cand['NODE'])

combined['MATCH_GROUP'] = 'ROW|' + combined.index.astype(str)   # default: its own group
combined.loc[cand.index, 'MATCH_GROUP'] = cand['MATCH_GROUP']

print(f"\nTotal rows: {len(combined):,}")
print(f"Rows with no usable address (own singleton group): {len(combined) - len(cand):,}")
print(f"Distinct MATCH_GROUP values: {combined['MATCH_GROUP'].nunique():,}")

Candidate rows: 2,035,982   Blocking groups: 562,012
Street-name pairs matched: 10,934
Street spellings merged into another spelling: 10,533

Total rows: 2,070,484
Rows with no usable address (own singleton group): 34,502
Distinct MATCH_GROUP values: 1,069,487


In [24]:
combined.to_pickle("../data/processed/nyc_sales_2003_2025_matched.pkl")
print("Saved:", combined.shape, "| has MATCH_GROUP:", "MATCH_GROUP" in combined.columns)

Saved: (2070484, 35) | has MATCH_GROUP: True


In [25]:
has_bbl = combined[combined['BBL'].notna() & (combined['BBL'] != 0)].copy()
has_bbl['UNIT_KEY'] = has_bbl['ADDRESS_UNIT'].fillna('<NONE>')

def n_pairs(sizes):
    s = sizes.astype('int64')
    return int((s * (s - 1) // 2).sum())

P  = n_pairs(has_bbl.groupby('MATCH_GROUP').size())                  # predicted same-property pairs
A  = n_pairs(has_bbl.groupby('BBL').size())                          # all same-BBL pairs
TP = n_pairs(has_bbl.groupby(['MATCH_GROUP', 'BBL']).size())         # predicted AND same BBL

hb = has_bbl[has_bbl['HOUSE_NUMBER'].notna()]
T  = n_pairs(hb.groupby(['BBL', 'HOUSE_NUMBER', 'UNIT_KEY']).size()) # fair truth set

print(f"Rows with a real BBL           : {len(has_bbl):,}")
print(f"Predicted pairs (P)            : {P:,}")
print(f"All same-BBL pairs (A)         : {A:,}")
print(f"True positives (TP)            : {TP:,}")
print(f"Fair truth-set pairs (T)       : {T:,}")
print(f"\nPrecision                      : {TP/P:.4f}")
print(f"Naive recall vs raw BBL (TP/A) : {TP/A:.4f}")
print(f"Recall vs fair truth (TP/T)    : {TP/T:.4f}")

Rows with a real BBL           : 814,658
Predicted pairs (P)            : 2,943,774
All same-BBL pairs (A)         : 19,582,970
True positives (TP)            : 2,932,759
Fair truth-set pairs (T)       : 3,055,812

Precision                      : 0.9963
Naive recall vs raw BBL (TP/A) : 0.1498
Recall vs fair truth (TP/T)    : 0.9597


In [26]:
keys = ['BBL', 'HOUSE_NUMBER', 'UNIT_KEY']

summ = hb.groupby(keys).agg(
    n=('MATCH_GROUP', 'size'),
    n_street=('STREET_NAME', 'nunique'),
    n_zip=('ZIP CODE', 'nunique'),
)
cell = hb.groupby(keys + ['MATCH_GROUP']).size()
tp_per_group = (cell * (cell - 1) // 2).groupby(level=[0, 1, 2]).sum()
summ['missed'] = (summ['n'] * (summ['n'] - 1) // 2) - tp_per_group.reindex(summ.index).fillna(0)

leaky = summ[summ['missed'] > 0]
same_street = leaky[leaky['n_street'] == 1]      # same street text, so the ZIP must differ
diff_street = leaky[leaky['n_street'] > 1]       # street spellings differ

print(f"Total missed pairs            : {int(leaky['missed'].sum()):,}   (should equal T - TP = {T - TP:,})")
print(f"  same street, different ZIP  : {int(same_street['missed'].sum()):,}  in {len(same_street):,} groups")
print(f"  different street spellings  : {int(diff_street['missed'].sum()):,}  in {len(diff_street):,} groups")

print("\n--- 15 examples of different-street misses (weighted by pairs missed) ---")
sample_keys = diff_street.sample(15, weights='missed', random_state=3).reset_index()[keys]
ex = hb.merge(sample_keys, on=keys)
for k, grp in ex.groupby(keys):
    print(k[1], '|', sorted(set(zip(grp['ZIP CODE'], grp['STREET_NAME']))))

Total missed pairs            : 123,053   (should equal T - TP = 123,053)
  same street, different ZIP  : 57,122  in 542 groups
  different street spellings  : 65,931  in 3,701 groups

--- 15 examples of different-street misses (weighted by pairs missed) ---


ValueError: Weighted sampling cannot be achieved with replace=False. Either set replace=True or use smaller weights. See the docstring of sample for details.

In [27]:
# 1) Different-street misses: the 6 biggest offenders + 10 random groups
top = diff_street.nlargest(6, 'missed')
rnd = diff_street.drop(top.index).sample(10, random_state=3)
sample_keys = pd.concat([top, rnd]).reset_index()[keys]
ex = hb.merge(sample_keys, on=keys)

print("--- Different-street misses ---")
for k, grp in ex.groupby(keys):
    print(f"missed={int(diff_street.loc[k, 'missed']):>6} | house {k[1]} unit {k[2]} |",
          sorted(set(zip(grp['ZIP CODE'], grp['STREET_NAME'])))[:6])

# 2) Same-street misses: is the ZIP missing/0, or are there two real ZIPs?
bad = hb['ZIP CODE'].isna() | (hb['ZIP CODE'] == 0)
bad_by_group = bad.groupby([hb['BBL'], hb['HOUSE_NUMBER'], hb['UNIT_KEY']]).any()
ss = same_street.join(bad_by_group.rename('has_missing_or_zero_zip'))
print("\n--- Same-street misses: is a ZIP missing or 0? ---")
print(ss.groupby('has_missing_or_zero_zip')['missed'].agg(['sum', 'count']))

--- Different-street misses ---
missed=    12 | house 111 unit <NONE> | [(10003.0, '3 AVE'), (10003.0, 'THIRD AVE')]
missed=     3 | house 234 unit <NONE> | [(10011.0, 'W 16 ST'), (10011.0, 'WEST 16 ST')]
missed=  1344 | house 252 unit <NONE> | [(10001.0, '7 AVE'), (10001.0, 'SEVENTH AVE')]
missed=  3223 | house 325 unit <NONE> | [(10016.0, '5 AVE'), (10016.0, '5 AVE S69'), (10016.0, 'FIFTH AVE')]
missed=  1667 | house 330 unit <NONE> | [(10016.0, 'E 38 ST'), (10016.0, 'E. 38 ST'), (10016.0, 'EAST 38 ST'), (10016.0, 'EAST 38TH STRET')]
missed=  3577 | house 630 unit <NONE> | [(10016.0, '1 AVE'), (10016.0, 'FIRST AVE')]
missed=  4959 | house 102 unit <NONE> | [(10019.0, 'W 57 ST'), (10019.0, 'WEST 57 ST')]
missed=  1413 | house 350 unit <NONE> | [(10036.0, 'W 42 ST'), (10036.0, 'WEST 42 ST')]
missed=     1 | house 167 unit <NONE> | [(10023.0, 'W 73 ST'), (10023.0, 'WEST 73 ST')]
missed=     1 | house 401 unit 2GH | [(10021.0, 'E. 74 ST'), (10021.0, 'EAST 74 ST')]
missed=     4 | house 4

In [28]:
import importlib, src.matching as m
importlib.reload(m)
for s in ['7 AVE', 'SEVENTH AVE', 'W 57 ST', 'WEST 57 ST', 'E. 38 ST', 'EAST 38TH STRET',
          'PARK AVE S', 'ST. PAUL AVE', 'FIFTH AVE', 'BROADWAY']:
    print(f"{s!r:22} -> {m.normalize_street(s)!r}")

'7 AVE'                -> '7 AVE'
'SEVENTH AVE'          -> '7 AVE'
'W 57 ST'              -> 'WEST 57 ST'
'WEST 57 ST'           -> 'WEST 57 ST'
'E. 38 ST'             -> 'EAST 38 ST'
'EAST 38TH STRET'      -> 'EAST 38 STRET'
'PARK AVE S'           -> 'PARK AVE SOUTH'
'ST. PAUL AVE'         -> 'ST PAUL AVE'
'FIFTH AVE'            -> '5 AVE'
'BROADWAY'             -> 'BROADWAY'


In [29]:
from itertools import combinations
from src.matching import street_similarity_match, UnionFind, normalize_street

uniq = combined['STREET_NAME'].dropna().unique()
combined['STREET_NORM'] = combined['STREET_NAME'].map({s: normalize_street(s) for s in uniq})

NO_UNIT = "<NONE>"
cand = combined[
    combined['HOUSE_NUMBER'].notna() & combined['STREET_NORM'].notna() & (combined['ZIP CODE'] != 0)
].copy()
cand['UNIT_KEY'] = cand['ADDRESS_UNIT'].fillna(NO_UNIT)
cand['ZIP_KEY']  = cand['ZIP CODE'].fillna(-1).astype(int)

grouped = cand.groupby(['ZIP_KEY', 'HOUSE_NUMBER', 'UNIT_KEY'])
print(f"Candidate rows: {len(cand):,}   Blocking groups: {grouped.ngroups:,}")

uf = UnionFind()
pairs_matched = 0
for key, g in grouped:
    streets = g['STREET_NORM'].unique()
    if len(streets) < 2:
        continue
    for a, b in combinations(streets, 2):
        if street_similarity_match(a, b):
            uf.union((key, a), (key, b))
            pairs_matched += 1
print(f"Street-name pairs matched: {pairs_matched:,}")

def node_str(node):
    (z, h, u), s = node
    return f"{z}|{h}|{u}|{s}"

mapping = {node_str(n): node_str(uf.find(n)) for n in list(uf.parent)}
cand['NODE'] = (cand['ZIP_KEY'].astype(str) + '|' + cand['HOUSE_NUMBER'] + '|'
                + cand['UNIT_KEY'] + '|' + cand['STREET_NORM'])
cand['MATCH_GROUP'] = cand['NODE'].map(mapping).fillna(cand['NODE'])

combined['MATCH_GROUP'] = 'ROW|' + combined.index.astype(str)
combined.loc[cand.index, 'MATCH_GROUP'] = cand['MATCH_GROUP']
print(f"Distinct MATCH_GROUP values: {combined['MATCH_GROUP'].nunique():,}")

Candidate rows: 2,035,982   Blocking groups: 562,012
Street-name pairs matched: 3,873
Distinct MATCH_GROUP values: 1,061,562


In [30]:
has_bbl = combined[combined['BBL'].notna() & (combined['BBL'] != 0)].copy()
has_bbl['UNIT_KEY'] = has_bbl['ADDRESS_UNIT'].fillna('<NONE>')

def n_pairs(sizes):
    s = sizes.astype('int64')
    return int((s * (s - 1) // 2).sum())

P  = n_pairs(has_bbl.groupby('MATCH_GROUP').size())                  # predicted same-property pairs
A  = n_pairs(has_bbl.groupby('BBL').size())                          # all same-BBL pairs
TP = n_pairs(has_bbl.groupby(['MATCH_GROUP', 'BBL']).size())         # predicted AND same BBL

hb = has_bbl[has_bbl['HOUSE_NUMBER'].notna()]
T  = n_pairs(hb.groupby(['BBL', 'HOUSE_NUMBER', 'UNIT_KEY']).size()) # fair truth set

print(f"Rows with a real BBL           : {len(has_bbl):,}")
print(f"Predicted pairs (P)            : {P:,}")
print(f"All same-BBL pairs (A)         : {A:,}")
print(f"True positives (TP)            : {TP:,}")
print(f"Fair truth-set pairs (T)       : {T:,}")
print(f"\nPrecision                      : {TP/P:.4f}")
print(f"Naive recall vs raw BBL (TP/A) : {TP/A:.4f}")
print(f"Recall vs fair truth (TP/T)    : {TP/T:.4f}")

Rows with a real BBL           : 814,658
Predicted pairs (P)            : 2,998,257
All same-BBL pairs (A)         : 19,582,970
True positives (TP)            : 2,987,225
Fair truth-set pairs (T)       : 3,055,812

Precision                      : 0.9963
Naive recall vs raw BBL (TP/A) : 0.1525
Recall vs fair truth (TP/T)    : 0.9776


In [31]:
combined.to_pickle("../data/processed/nyc_sales_2003_2025_matched.pkl")
print("Saved:", combined.shape)

Saved: (2070484, 36)


In [32]:
from itertools import combinations
from src.matching import street_similarity_match, UnionFind

NO_UNIT = "<NONE>"

def nid(boro, z, house, unit, street):
    return f"{boro}|{z}|{house}|{unit}|{street}"

cand = combined[combined['HOUSE_NUMBER'].notna() & combined['STREET_NORM'].notna()].copy()
cand['UNIT_KEY'] = cand['ADDRESS_UNIT'].fillna(NO_UNIT)
cand['ZIP_KEY']  = cand['ZIP CODE'].where(cand['ZIP CODE'] > 0).fillna(-1).astype(int)   # missing or 0 -> -1
print(f"Candidate rows: {len(cand):,}   of which missing/0 ZIP: {(cand['ZIP_KEY'] == -1).sum():,}")

uf = UnionFind()

# Pass 1: same borough + ZIP + house number + unit, fuzzy street match
for (boro, z, house, unit), g in cand.groupby(['BOROUGH', 'ZIP_KEY', 'HOUSE_NUMBER', 'UNIT_KEY']):
    streets = g['STREET_NORM'].unique()
    if len(streets) < 2:
        continue
    for a, b in combinations(streets, 2):
        if street_similarity_match(a, b):
            uf.union(nid(boro, z, house, unit, a), nid(boro, z, house, unit, b))
print(f"Pass 1 nodes involved in merges: {len(uf.parent):,}")

# Pass 2: rescue missing-ZIP rows, only where exactly one real-ZIP node matches
miss_keys = cand.loc[cand['ZIP_KEY'] == -1, ['BOROUGH', 'HOUSE_NUMBER', 'UNIT_KEY']].drop_duplicates()
sub = cand.merge(miss_keys, on=['BOROUGH', 'HOUSE_NUMBER', 'UNIT_KEY'])

rescued = ambiguous = no_candidate = 0
for (boro, house, unit), g in sub.groupby(['BOROUGH', 'HOUSE_NUMBER', 'UNIT_KEY']):
    nodes = g[['ZIP_KEY', 'STREET_NORM']].drop_duplicates()
    miss_nodes = nodes[nodes['ZIP_KEY'] == -1]['STREET_NORM'].tolist()
    real_nodes = list(nodes[nodes['ZIP_KEY'] != -1].itertuples(index=False))
    for ms in miss_nodes:
        hits = [(r.ZIP_KEY, r.STREET_NORM) for r in real_nodes
                if street_similarity_match(ms, r.STREET_NORM)]
        if len(hits) == 1:
            uf.union(nid(boro, -1, house, unit, ms), nid(boro, hits[0][0], house, unit, hits[0][1]))
            rescued += 1
        elif len(hits) > 1:
            ambiguous += 1
        else:
            no_candidate += 1
print(f"Missing-ZIP street nodes -> rescued: {rescued:,} | ambiguous (left alone): {ambiguous:,} | no match: {no_candidate:,}")

mapping = {n: uf.find(n) for n in list(uf.parent)}
cand['NODE'] = (cand['BOROUGH'].astype(str) + '|' + cand['ZIP_KEY'].astype(str) + '|'
                + cand['HOUSE_NUMBER'] + '|' + cand['UNIT_KEY'] + '|' + cand['STREET_NORM'])
cand['MATCH_GROUP'] = cand['NODE'].map(mapping).fillna(cand['NODE'])

combined['MATCH_GROUP'] = 'ROW|' + combined.index.astype(str)
combined.loc[cand.index, 'MATCH_GROUP'] = cand['MATCH_GROUP']
print(f"Distinct MATCH_GROUP values: {combined['MATCH_GROUP'].nunique():,}")

Candidate rows: 2,037,786   of which missing/0 ZIP: 1,822
Pass 1 nodes involved in merges: 7,412
Missing-ZIP street nodes -> rescued: 395 | ambiguous (left alone): 121 | no match: 280
Distinct MATCH_GROUP values: 1,060,139


In [34]:
## step 18 

has_bbl = combined[combined['BBL'].notna() & (combined['BBL'] != 0)].copy()
has_bbl['UNIT_KEY'] = has_bbl['ADDRESS_UNIT'].fillna('<NONE>')

def n_pairs(sizes):
    s = sizes.astype('int64')
    return int((s * (s - 1) // 2).sum())

P  = n_pairs(has_bbl.groupby('MATCH_GROUP').size())                  # predicted same-property pairs
A  = n_pairs(has_bbl.groupby('BBL').size())                          # all same-BBL pairs
TP = n_pairs(has_bbl.groupby(['MATCH_GROUP', 'BBL']).size())         # predicted AND same BBL

hb = has_bbl[has_bbl['HOUSE_NUMBER'].notna()]
T  = n_pairs(hb.groupby(['BBL', 'HOUSE_NUMBER', 'UNIT_KEY']).size()) # fair truth set

print(f"Rows with a real BBL           : {len(has_bbl):,}")
print(f"Predicted pairs (P)            : {P:,}")
print(f"All same-BBL pairs (A)         : {A:,}")
print(f"True positives (TP)            : {TP:,}")
print(f"Fair truth-set pairs (T)       : {T:,}")
print(f"\nPrecision                      : {TP/P:.4f}")
print(f"Naive recall vs raw BBL (TP/A) : {TP/A:.4f}")
print(f"Recall vs fair truth (TP/T)    : {TP/T:.4f}")

Rows with a real BBL           : 814,658
Predicted pairs (P)            : 3,037,817
All same-BBL pairs (A)         : 19,582,970
True positives (TP)            : 3,026,782
Fair truth-set pairs (T)       : 3,055,812

Precision                      : 0.9964
Naive recall vs raw BBL (TP/A) : 0.1546
Recall vs fair truth (TP/T)    : 0.9905


In [35]:
combined.to_pickle("../data/processed/nyc_sales_2003_2025_matched.pkl")
print("Saved:", combined.shape)

Saved: (2070484, 36)


In [36]:
# --- 1) Precision / recall by borough ---
names = {1: 'Manhattan', 2: 'Bronx', 3: 'Brooklyn', 4: 'Queens', 5: 'Staten Island'}
rows = []
for b, d in has_bbl.groupby('BOROUGH'):
    P_b  = n_pairs(d.groupby('MATCH_GROUP').size())
    TP_b = n_pairs(d.groupby(['MATCH_GROUP', 'BBL']).size())
    dh   = d[d['HOUSE_NUMBER'].notna()]
    T_b  = n_pairs(dh.groupby(['BBL', 'HOUSE_NUMBER', 'UNIT_KEY']).size())
    rows.append({'borough': names.get(b, b), 'rows': len(d), 'precision': TP_b / P_b, 'recall_fair': TP_b / T_b})
print(pd.DataFrame(rows).round(4).to_string(index=False))

# --- 2) False positives: predicted-same pairs whose BBLs differ ---
mg   = has_bbl.groupby('MATCH_GROUP').size()
cell = has_bbl.groupby(['MATCH_GROUP', 'BBL']).size()
tp_g = (cell * (cell - 1) // 2).groupby(level=0).sum()
fp   = ((mg * (mg - 1) // 2) - tp_g.reindex(mg.index).fillna(0))
fp   = fp[fp > 0].sort_values(ascending=False)
print(f"\nGroups containing false-positive pairs: {len(fp):,}   (total FP pairs: {int(fp.sum()):,})")

keys = list(fp.index[:6]) + list(fp.iloc[6:].sample(6, random_state=5).index)
for k in keys:
    grp = has_bbl[has_bbl['MATCH_GROUP'] == k]
    print(f"\n{k}   | FP pairs: {int(fp[k])}")
    for bbl, r in grp.groupby('BBL')['SALE DATE'].agg(['size', 'min', 'max']).iterrows():
        print(f"    BBL {bbl} | rows {r['size']} | {r['min'].date()} -> {r['max'].date()}")

      borough   rows  precision  recall_fair
    Manhattan 179784     0.9968       0.9928
        Bronx  68014     0.9991       0.9878
     Brooklyn 231890     0.9973       0.9784
       Queens 255924     0.9935       0.9982
Staten Island  79046     0.9978       0.9938

Groups containing false-positive pairs: 737   (total FP pairs: 11,035)

4|11101|24-16|<NONE>|QUEENS PLAZA SOUTH   | FP pairs: 2303
    BBL 4004240019 | rows 49 | 2016-03-11 -> 2021-12-22
    BBL 4004247501 | rows 47 | 2020-04-02 -> 2020-12-14

1|10038|25|<NONE>|PARK ROW   | FP pairs: 2052
    BBL 1000900007 | rows 38 | 2021-01-20 -> 2021-12-30
    BBL 1000907504 | rows 54 | 2020-07-13 -> 2020-12-30

1|10014|110|<NONE>|CHARLTON ST   | FP pairs: 1281
    BBL 1005970039 | rows 21 | 2020-10-19 -> 2020-12-29
    BBL 1005970046 | rows 61 | 2021-03-11 -> 2021-12-20

1|10007|49-51|<NONE>|CHAMBERS ST   | FP pairs: 648
    BBL 1001530018 | rows 36 | 2021-01-15 -> 2021-12-30
    BBL 1001537501 | rows 18 | 2020-01-03 -> 2020-12-29


In [37]:
has_bbl['IS_BILLING'] = (has_bbl['BBL'] % 10000) >= 7501

flags = has_bbl.groupby('MATCH_GROUP')['IS_BILLING'].agg(['any', 'all'])
mixed_groups = flags[flags['any'] & ~flags['all']].index      # groups mixing billing-lot and regular-lot BBLs

fp_mixed = int(fp[fp.index.isin(mixed_groups)].sum())
print(f"Total false-positive pairs               : {int(fp.sum()):,}")
print(f"...in groups mixing billing + regular BBL : {fp_mixed:,}  ({fp_mixed / fp.sum():.1%})")
print(f"...in all other groups                    : {int(fp.sum()) - fp_mixed:,}")

Total false-positive pairs               : 11,035
...in groups mixing billing + regular BBL : 8,621  (78.1%)
...in all other groups                    : 2,414


# Phase 2 Summary: Entity Resolution

**Goal:** identify which sale records refer to the same physical property, treating BBL as
an answer key to validate against rather than as an input.

## Method
1. Normalize addresses (case, whitespace, street-type abbreviations, ordinal suffixes).
2. Decompose each address into house number, street name, and unit. Numeric tokens at every
   level (house number, street number, unit) require EXACT match, because a one-digit
   difference means a different place (1G vs 11G, 2116 vs 211, EAST 87 ST vs EAST 81 ST).
3. Normalize street names (spelled-out ordinals FIRST->1, directionals W->WEST, ordinal suffixes).
4. Block on borough + ZIP + house number + unit; fuzzy-match street names within blocks
   (rapidfuzz token_sort_ratio >= 90) and merge chains with union-find.
5. Rescue rows with a missing ZIP, only when exactly one real-ZIP candidate matches
   (ambiguous cases are left unmatched on purpose). Block/lot/BBL were never used to match.

## Validation (rows with a real BBL: 814,658; pairwise evaluation)
| Metric | Value |
|---|---|
| Precision vs BBL (strict) | 99.64% (3,026,782 of 3,037,817 predicted pairs) |
| Precision, adjusting for condo billing-lot BBLs | up to ~99.92% (78% of apparent errors are the same building under two BBL types) |
| Recall vs fair truth (same BBL + house number + unit) | 99.05% (3,026,782 of 3,055,812 pairs) |
| Naive recall vs raw BBL | 15.5%, misleading: one BBL often covers a whole multi-unit parcel |

By borough: precision 99.35%-99.91%, recall 97.84% (Brooklyn) to 99.82% (Queens).

## Iterative validation (each fix found by inspecting real output)
- Fuzzy scoring treated unit/house/street numbers as similar strings -> exact-match rule.
- Apartment text without a comma stayed inside the street name -> splitter fixed.
- "UNITED" parsed as UNIT + "ED" (1,117 rows) -> whole-word matching.
- Recall: 95.97% -> 97.76% (street normalization) -> 99.05% (missing-ZIP rescue).

## Limitations
- BBL ground truth exists only for 2016+; pre-2016 rows use the same logic, validated indirectly.
- 34,502 rows have no usable address and stay unmatched by address.
- Unit-less condo sales cannot be told apart from the address alone.
- ~0.95% of true pairs missed (ambiguous ZIPs, truncated street names).
- Fuzzy threshold 90 was chosen from inspecting examples, not tuned by a sweep.
- 805 units are literally "N/A"; they are currently treated as unit names.